In [1]:
import polars as pl
from Python import statcast, ballpark

years = [2023, 2024, 2025]

In [2]:
print(ballpark.DEFAULT_REGRESS_PA)

500.0


In [3]:
raw = statcast.load_statcast_years(years)
pa = statcast.plate_appearances(raw)
pa.shape

(550203, 20)

In [4]:
factors_all = ballpark.park_k_factor(pa)
factors_all

# Sanity: league-average factor should sit very close to 1.0 when weighted by PA
weighted_check = (factors_all["park_k_factor"] * factors_all["PA"]).sum() / factors_all["PA"].sum()
weighted_check

0.9999540114681164

In [5]:
# Small-PA parks should sit closer to 1.0 than large-PA parks with the same raw K rate
factors_all.sort("PA").select(["venue", "PA", "K", "park_k_factor"]).head(5)
factors_all.sort("PA", descending=True).select(["venue", "PA", "K", "park_k_factor"]).head(5)

venue,PA,K,park_k_factor
str,u32,u32,f64
"""COL""",19118,3899,0.909807
"""BOS""",18706,4115,0.979366
"""AZ""",18646,3845,0.919687
"""MIA""",18596,4020,0.962871
"""LAA""",18474,4387,1.055126


In [6]:
target_seasons = [*years, 2026]
pregame = ballpark.pregame_park_factors(pa, target_seasons=target_seasons)
print(pregame.shape)
pregame.head(10)

(120, 5)


season,home_team,PA,K,park_k_factor
i32,str,u32,u32,f64
2023,"""ATL""",0,0,1.0
2023,"""AZ""",0,0,1.0
2023,"""BAL""",0,0,1.0
2023,"""BOS""",0,0,1.0
2023,"""CHC""",0,0,1.0
2023,"""CIN""",0,0,1.0
2023,"""CLE""",0,0,1.0
2023,"""COL""",0,0,1.0
2023,"""CWS""",0,0,1.0


In [7]:
first_season = min(years)
first_season_rows = pregame.filter(pl.col("season") == first_season)

first_season_rows.select(
    (pl.col("park_k_factor") == 1.0).all().alias("all_neutral"),
    (pl.col("PA") == 0).all().alias("all_zero_pa"),
)

all_neutral,all_zero_pa
bool,bool
true,true


In [8]:
later_season = max(years)
later_rows = (
    pregame.filter(pl.col("season") == later_season)
    .select(["home_team", "park_k_factor"])
    .with_columns(pl.date(later_season, 7, 1).alias("game_date"))
)
later_rows = ballpark._resolve_venue(later_rows)
full_rows = factors_all.select(["venue", "park_k_factor"]).rename(
    {"park_k_factor": "park_k_factor_full"}
)

compare = later_rows.join(full_rows, on="venue", how="left")
compare.with_columns(
    (pl.col("park_k_factor") != pl.col("park_k_factor_full")).alias("differs_from_full_history")
)

home_team,park_k_factor,game_date,venue,park_k_factor_full,differs_from_full_history
str,f64,date,str,f64,bool
"""ATH""",1.0,2025-07-01,"""ATH""",0.979184,true
"""ATL""",1.06775,2025-07-01,"""ATL""",1.059482,true
"""AZ""",0.923657,2025-07-01,"""AZ""",0.919687,true
"""BAL""",0.980225,2025-07-01,"""BAL""",0.998922,true
"""BOS""",0.993142,2025-07-01,"""BOS""",0.979366,true
…,…,…,…,…,…
"""STL""",0.901627,2025-07-01,"""STL""",0.898268,true
"""TB""",1.0,2025-07-01,"""TB_steinbrenner""",1.02502,true
"""TEX""",1.001057,2025-07-01,"""TEX""",1.008045,true


In [9]:
coverage = (
    pregame.group_by("season")
    .agg(pl.col("home_team").n_unique().alias("teams_covered"))
    .sort("season")
)
coverage

# Every MLB season should have one lookup row for each of 30 teams, including
# future target seasons that have no Statcast rows yet.
coverage.filter(pl.col("teams_covered") != 30)

season,teams_covered
i32,u32


In [10]:
pregame.filter(pl.col("home_team") == "TB").sort("season")

season,home_team,PA,K,park_k_factor
i32,str,u32,u32,f64
2023,"""TB""",0,0,1.0
2024,"""TB""",6019,1522,1.105462
2025,"""TB""",0,0,1.0
2026,"""TB""",12064,2987,1.097633


In [11]:
# Venue/code verification
athletics_home_games = (
    raw.filter(
        (pl.col("game_date").dt.year() == 2025)
        & (pl.col("home_team") == "ATH")
    )["game_pk"].n_unique()
)
print("2025 ATH home games:", athletics_home_games)
print("2026 Statcast file exists:", statcast.season_path(2026).exists())
print("2026 park lookup rows:", pregame.filter(pl.col("season") == 2026).height)

2025 ATH home games: 81
2026 Statcast file exists: False
2026 park lookup rows: 30
